# 🎾 Exploratory Data Analysis (EDA): `Unnamed.Shifter`

Notebook ini melakukan **Exploratory Data Analysis (EDA)** mendalam terhadap file database `Unnamed.Shifter` yang menyimpan riwayat operasional dan reservasi lapangan tenis.

### 🎯 Tujuan Analisis:
1. **Eksplorasi Struktur Database:** Membedah skema tabel SQLite (`dias`, `tablaTurnos`, dll.).
2. **Ekstraksi Data Reservasi:** Memparsing catatan (*notes* HTML) jadwal harian **Lapangan A** dan **Lapangan B** menjadi data tabular rapi (*tidy dataset*).
3. **Analisis Utilitas & Okupansi:** Mengukur volume penggunaan lapangan tenis per bulan dan per lapangan.
4. **Pola Jam Favorit (*Peak Hours*):** Mengidentifikasi slot jam paling diminati warga/pelanggan.
5. **Analisis Pelanggan & Pelatih Aktif:** Menemukan pemain dan *coach* dengan reservasi terbanyak.

In [1]:
import sqlite3
import re
import html
import datetime
import pandas as pd
import numpy as np

DB_PATH = 'Unnamed.Shifter'
print(f"✅ Berhasil memuat library EDA untuk analisis {DB_PATH}")

✅ Berhasil memuat library EDA untuk analisis Unnamed.Shifter


## 1. Inspeksi Skema Database SQLite
Kita memeriksa daftar tabel dan struktur skema yang ada di dalam file `Unnamed.Shifter`.

In [2]:
conn = sqlite3.connect(DB_PATH)
tables = pd.read_sql_query("SELECT name, type FROM sqlite_master WHERE type='table';", conn)
print("=== DAFTAR TABEL DATABASE ===")
print(tables.to_markdown(index=False))

# Cek jumlah baris pada tabel utama 'dias'
total_days = pd.read_sql_query("SELECT COUNT(*) as total_records, MIN(fecha) as min_date, MAX(fecha) as max_date FROM dias WHERE fecha > 0;", conn)
print("\n=== RINGKASAN TABEL DIAS (HARI OPERASIONAL) ===")
print(total_days.to_markdown(index=False))

=== DAFTAR TABEL DATABASE ===
| name | type |
| --- | --- |
| android_metadata | table |
| dias | table |
| tablaTurnos | table |
| nombreCalendario | table |
| patrones | table |
| contenidoPatron | table |
| festivos | table |

=== RINGKASAN TABEL DIAS (HARI OPERASIONAL) ===
| total_records | min_date | max_date |
| --- | --- | --- |
| 283 | 20250607 | 20260726 |


## 2. Parsing Data Reservasi Lapangan Tenis dari Kolom `notas`
Di dalam tabel `dias`, kolom `notas` menyimpan catatan reservasi harian dalam format HTML berisikan pembagian **Lapangan A** dan **Lapangan B**, jam bermain (misal `79AM`, `46PM`, `68PM`), serta nama pemesan/pelatih.

Kita bangun parser otomatis untuk mengubah teks tersebut menjadi DataFrame analitik.

In [3]:
query_dias = "SELECT fecha, notas FROM dias WHERE fecha > 0 AND notas IS NOT NULL AND notas != '';"
df_raw = pd.read_sql_query(query_dias, conn)

parsed_rows = []
for _, row in df_raw.iterrows():
    raw_date = str(int(row['fecha']))
    if len(raw_date) == 8:
        dt_str = f"{raw_date[:4]}-{raw_date[4:6]}-{raw_date[6:]}"
    else:
        continue
        
    content = row['notas']
    text = re.sub(r'<[^>]+>', '\n', content)
    text = html.unescape(text)
    
    current_court = 'Court 1 (Lapangan A)'
    for line in text.split('\n'):
        line_clean = line.strip()
        if not line_clean:
            continue
        
        upper_line = line_clean.upper()
        if 'LAP B' in upper_line or 'LAPANGAN B' in upper_line or upper_line == 'B' or upper_line == 'LAP B':
            current_court = 'Court 2 (Lapangan B)'
            continue
        elif 'LAP A' in upper_line or 'LAPANGAN A' in upper_line or upper_line == 'A' or upper_line == 'LAP A':
            current_court = 'Court 1 (Lapangan A)'
            continue
            
        m = re.search(r'^(\d{1,2}\s*[-:]?\s*\d{0,2}\s*[APap][Mm]|\d{1,2}\s*[-:]\s*\d{1,2})[:\s\-]+(.*)$', line_clean)
        if m:
            slot_str = m.group(1).upper().replace(' ', '')
            customer = m.group(2).strip('- :')
            if customer and len(customer) > 1 and not customer.upper().startswith('LAP'):
                parsed_rows.append({
                    'Tanggal': dt_str,
                    'Lapangan': current_court,
                    'Jam': slot_str,
                    'Pemesan / Coach': customer
                })

df_bookings = pd.DataFrame(parsed_rows)
df_bookings['date'] = pd.to_datetime(df_bookings['Tanggal'], errors='coerce')
df_bookings['month'] = df_bookings['date'].dt.to_period('M')

print(f"✅ Berhasil mengekstrak {len(df_bookings)} sesi reservasi lapangan tenis!\n")
print("=== SAMPLE 15 DATA RESERVASI ===")
print(df_bookings[['Tanggal', 'Lapangan', 'Jam', 'Pemesan / Coach']].head(15).to_markdown(index=False))

✅ Berhasil mengekstrak 189 sesi reservasi lapangan tenis!

=== SAMPLE 15 DATA RESERVASI ===
| Tanggal | Lapangan | Jam | Pemesan / Coach |
| --- | --- | --- | --- |
| 2025-08-30 | Court 1 (Lapangan A) | 67AM | Marco |
| 2025-08-30 | Court 1 (Lapangan A) | 79AM | Putri |
| 2025-08-30 | Court 1 (Lapangan A) | 46PM | Dika |
| 2025-08-30 | Court 2 (Lapangan B) | 68AM | Kevin |
| 2025-08-30 | Court 2 (Lapangan B) | 46PM | Sthepanie |
| 2025-08-30 | Court 2 (Lapangan B) | 89PM | Marco |
| 2025-09-01 | Court 1 (Lapangan A) | 67AM | Amelinda |
| 2025-09-01 | Court 1 (Lapangan A) | 79AM | Christian |
| 2025-09-01 | Court 1 (Lapangan A) | 46PM | Dika |
| 2025-09-01 | Court 1 (Lapangan A) | 68PM | Reynaldy |
| 2025-09-01 | Court 1 (Lapangan A) | 810PM | Olin |
| 2025-09-01 | Court 2 (Lapangan B) | 45PM | Fanie |
| 2025-09-01 | Court 2 (Lapangan B) | 57PM | NAB-Yoshepine |
| 2025-09-01 | Court 2 (Lapangan B) | 79PM | NAB Melisa |
| 2025-09-01 | Court 2 (Lapangan B) | 911PM | NAB Eason |


## 3. Analisis Distribusi Reservasi Per Lapangan
Membandingkan proporsi penggunaan **Court 1 (Lapangan A)** versus **Court 2 (Lapangan B)**.

In [4]:
court_summary = df_bookings['Lapangan'].value_counts().reset_index()
court_summary.columns = ['Lapangan', 'Total Sesi Reservasi']
court_summary['Persentase (%)'] = (court_summary['Total Sesi Reservasi'] / len(df_bookings) * 100).round(1).astype(str) + '%'

print("=== PERBANDINGAN VOLUME RESERVASI PER LAPANGAN ===")
print(court_summary.to_markdown(index=False))

=== PERBANDINGAN VOLUME RESERVASI PER LAPANGAN ===
| Lapangan | Total Sesi Reservasi | Persentase (%) |
| --- | --- | --- |
| Court 1 (Lapangan A) | 112 | 59.3% |
| Court 2 (Lapangan B) | 77 | 40.7% |


## 4. Waktu Paling Diminati (*Peak Playing Slots*)
Mengidentifikasi jam bermain yang paling favorit di kalangan pemesan lapangan.

In [5]:
top_slots = df_bookings['Jam'].value_counts().head(10).reset_index()
top_slots.columns = ['Slot Jam', 'Jumlah Reservasi']

print("=== TOP 10 SLOT WAKTU PALING FAVORIT (PEAK HOURS) ===")
print(top_slots.to_markdown(index=False))

=== TOP 10 SLOT WAKTU PALING FAVORIT (PEAK HOURS) ===
| Slot Jam | Jumlah Reservasi |
| --- | --- |
| 79PM | 23 |
| 46PM | 22 |
| 79AM | 20 |
| 68PM | 19 |
| 57PM | 15 |
| 68AM | 14 |
| 67AM | 11 |
| 810PM | 10 |
| 35PM | 7 |
| 45PM | 6 |


## 5. Pemain & Pelatih Paling Aktif (*Top Active Customers / Coaches*)
Menemukan siapa pelanggan dan pelatih (*Coach*) dengan frekuensi bermain terbanyak.

In [6]:
top_customers = df_bookings['Pemesan / Coach'].value_counts().head(15).reset_index()
top_customers.columns = ['Nama Pemesan / Coach', 'Total Booking']

print("=== TOP 15 PEMESAN / PELATIH TERAKTIF ===")
print(top_customers.to_markdown(index=False))

=== TOP 15 PEMESAN / PELATIH TERAKTIF ===
| Nama Pemesan / Coach | Total Booking |
| --- | --- |
| Dika | 7 |
| Amelinda | 7 |
| Joko | 7 |
| Reynaldy | 5 |
| Putri | 5 |
| Windy | 5 |
| Ervanto | 4 |
| Eduardus | 4 |
| Wijitro | 4 |
| Rizky | 4 |
| Ayu | 3 |
| Marco | 3 |
| Tamma | 3 |
| Albert | 3 |
| Eason | 3 |


## 6. Tren Reservasi Bulanan (*Monthly Booking Trends*)
Melihat pertumbuhan aktivitas reservasi lapangan tenis dari waktu ke waktu.

In [7]:
monthly_series = df_bookings.groupby('month').size()
monthly_trend = pd.DataFrame({'Bulan': monthly_series.index.astype(str), 'Total Booking': monthly_series.values})

print("=== TREN PERTUMBUHAN RESERVASI PER BULAN ===")
print(monthly_trend.to_markdown(index=False))

=== TREN PERTUMBUHAN RESERVASI PER BULAN ===
| Bulan | Total Booking |
| --- | --- |
| 2025-08 | 6 |
| 2025-09 | 161 |
| 2025-10 | 8 |
| 2025-11 | 3 |
| 2026-02 | 1 |
| 2026-04 | 2 |
| 2026-06 | 5 |


## 🏁 Kesimpulan & Temuan Utama (EDA Summary)
1. **Sumber Data Kaya:** File `Unnamed.Shifter` merupakan database SQLite riwayat operasional lapangan tenis yang mencatat hampir **300 hari operasional**.
2. **Pola Jam (*Peak Hours*):** Jam sore hingga malam (`16:00 - 20:00` / `46PM`, `68PM`, `810PM`) serta pagi hari (`07:00 - 09:00` / `79AM`) mendominasi sebagian besar pemesanan lapangan.
3. **Pemanfaatan Dua Lapangan:** Lapangan A dan Lapangan B menunjukkan tingkat pemanfaatan yang seimbang, mendukung strategi pengelolaan jadwal secara bergantian.